# LoRA 가중치 HuggingFace 업로드

저장된 LoRA 가중치(`joseon_lora/`)를 HuggingFace Hub에 업로드하고,  
다시 다운로드해서 정상 동작하는지 확인합니다.

> ⚠️ 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행하세요.

## 1. 패키지 설치

In [ ]:
!pip install -q --upgrade transformers peft bitsandbytes accelerate
!pip install -q huggingface_hub

## 2. HuggingFace 로그인

[https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) 에서 **Write 권한** 토큰을 생성한 뒤, 아래 셀 실행 시 입력하세요.

In [ ]:
from huggingface_hub import login
login()

## 3. 저장된 LoRA 가중치 확인

`joseon_lora/` 폴더에 어떤 파일들이 저장되어 있는지 확인합니다.

In [ ]:
import os

LORA_DIR = "joseon_lora"

print(f"--- {LORA_DIR}/ 저장된 파일 ---")
for f in sorted(os.listdir(LORA_DIR)):
    size = os.path.getsize(os.path.join(LORA_DIR, f))
    if size > 1024 * 1024:
        print(f"  {f:40s} {size/1024/1024:.1f} MB")
    else:
        print(f"  {f:40s} {size/1024:.1f} KB")

## 4. HuggingFace Hub에 업로드

`HfApi.upload_folder()`로 `joseon_lora/` 폴더를 통째로 업로드합니다.

`REPO_ID`를 본인의 HuggingFace 사용자명으로 변경하세요.  
예: `"my-username/joseon-qwen2.5-1.5b-lora"`

### 모델 카드 작성

HuggingFace 레포지토리에 표시되는 모델 설명(README.md)을 작성합니다.  
`joseon_lora/` 폴더 안에 `README.md`를 만들어두면 업로드 시 자동으로 함께 올라갑니다.

In [ ]:
import os

model_card = """---
language: ko
license: apache-2.0
tags:
  - lora
  - qwen2.5
  - fine-tuning
base_model: Qwen/Qwen2.5-1.5B-Instruct
---

# Joseon Style Chatbot — LoRA Adapter

Qwen2.5-1.5B-Instruct를 조선시대 말투로 대화하도록 QLoRA로 fine-tuning한 어댑터입니다.

## 모델 정보

| 항목 | 내용 |
|------|------|
| 베이스 모델 | `Qwen/Qwen2.5-1.5B-Instruct` |
| 학습 방법 | QLoRA (4bit, HuggingFace + bitsandbytes + PEFT) |
| LoRA r | 16 |
| LoRA alpha | 16 |
| 학습 환경 | Google Colab (T4 GPU) |

## 사용법

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = PeftModel.from_pretrained(base_model, "YOUR_REPO_ID")
```
"""

with open(os.path.join(LORA_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(model_card)

print("✅ 모델 카드(README.md) 생성 완료")
print(f"   → {LORA_DIR}/README.md")

In [ ]:
from huggingface_hub import HfApi

REPO_ID = "my-username/joseon-qwen2.5-1.5b-lora"  # ← 본인 사용자명으로 변경!

api = HfApi()

# 레포지토리 생성 (이미 있으면 무시)
api.create_repo(REPO_ID, private=False, exist_ok=True)

# 폴더 통째로 업로드
api.upload_folder(
    folder_path=LORA_DIR,
    repo_id=REPO_ID,
)

print(f"✅ 업로드 완료! → https://huggingface.co/{REPO_ID}")

## 5. 업로드된 모델 다운로드 & 테스트

HuggingFace에서 방금 업로드한 LoRA 가중치를 다시 받아와서 정상 동작하는지 확인합니다.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# 1) 베이스 모델 로드 (학습할 때와 동일한 설정)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2) HuggingFace에서 LoRA 가중치 다운로드 & 적용
model = PeftModel.from_pretrained(base_model, REPO_ID)

print("✅ HuggingFace에서 모델 로드 완료!")

In [ ]:
def generate(prompt, model, max_new_tokens=200):
    model.eval()
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=False,
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
        )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

test_questions = [
    "오늘 점심 뭐 먹을까?",
    "요즘 너무 피곤해.",
    "주말에 어디 가면 좋을까?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {generate(q, model)}")
    print("-" * 50)

print("\n✅ HuggingFace에서 다운로드한 모델이 정상 동작합니다!")

## 6. 다음 단계 — 추론/서빙 옵션

Hub 업로드는 "모델을 공유 가능한 형태로 저장"한 것일 뿐, 아직 **서비스**는 아닙니다.
업로드한 모델을 실제로 쓰는 대표적인 방법 두 가지를 소개합니다.

### ① Gradio + HuggingFace Spaces — 데모 웹앱 (가장 추천)

채팅 UI를 몇 줄로 만들어 누구나 접속할 수 있는 웹 데모로 배포합니다.

```python
import gradio as gr

def chat(message, history):
    return generate(message, model)

gr.ChatInterface(chat).launch()   # Colab에서는 share=True 로 임시 공개 링크 생성
```

- [huggingface.co/spaces](https://huggingface.co/spaces) → **Create new Space** → SDK: **Gradio** 선택
- 위 코드와 모델 로드 코드를 `app.py`로 올리면 무료로 호스팅됩니다.
- 학습 결과를 바로 눈으로 확인할 수 있어 성취감이 큽니다.

### ② GGUF 변환 + Ollama / llama.cpp — 로컬 실행

GPU 없이 **개인 PC(CPU)에서도** 돌릴 수 있는 경량 포맷(GGUF)으로 변환합니다.

1. **어댑터 병합**: LoRA 어댑터를 베이스 모델에 합쳐 단일 모델로 만듭니다 (`merge_and_unload`).
2. **GGUF 변환**: [`llama.cpp`](https://github.com/ggerganov/llama.cpp)의 `convert_hf_to_gguf.py`로 변환 + 양자화(예: `q4_k_m`).
3. **실행**:
   ```bash
   ollama create joseon -f Modelfile   # Modelfile에 GGUF 경로 지정
   ollama run joseon
   ```

| 방법 | 환경 | 용도 |
|------|------|------|
| Gradio + Spaces | 클라우드(무료) | 데모 공유, 발표 |
| GGUF + Ollama | 로컬 PC(CPU 가능) | 오프라인 실행, 개인용 |

> 그 밖에 프로덕션 규모 서빙에는 **vLLM**, **TGI**, **HF Inference Endpoints** 등을 사용합니다.